In [1]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from scipy.stats import chi2


from itertools import product
import os
import h5py
from tqdm import tqdm
import pandas as pd

#few utils
from few.utils.utility import get_p_at_t
from few.utils.constants import MTSUN_SI
from few.utils.geodesic import get_fundamental_frequencies
#few trajectory
from few.trajectory.inspiral import EMRIInspiral
from few.trajectory.ode.flux import SuperKludgeFlux
#few waveform
from few.waveform import FastKerrEccentricEquatorialFlux, GenerateEMRIWaveform
from few.waveform.waveform import SuperKludgeWaveform
from few.utils.constants import YRSID_SI

#sef imports
from stableemrifisher.fisher import StableEMRIFisher
from stableemrifisher.utils import generate_PSD, padding, inner_product
from stableemrifisher.fisher.derivatives import derivative
from stableemrifisher.fisher.stablederivative import StableEMRIDerivative
from stableemrifisher.noise import sensitivity_LWA

#lisa-on-gpu import
from fastlisaresponse import ResponseWrapper  # Response function 

#LISAanalysistools imports
from lisatools.detector import ESAOrbits, EqualArmlengthOrbits #ESAOrbits correspond to esa-trailing-orbits.h5, EqualArmlengthOrbits are equalarmlength-orbits.h5
from lisatools.sensitivity import get_sensitivity, A1TDISens, E1TDISens, T1TDISens
from lisatools.sensitivity import get_sensitivity,CornishLISASens

use_gpu = True
from parismc.sampler import SamplerConfig
from parismc.sampler import Sampler

from smt.sampling_methods import LHS

if not use_gpu:
    
    import few
    
    #tune few configuration
    cfg_set = few.get_config_setter(reset=True)
    
    cfg_set.enable_backends("cpu")
    cfg_set.set_log_level("info");
else:
    pass #let the backend decide for itself

startup


In [2]:
#waveform class setup
waveform_class = SuperKludgeWaveform
max_step_days = 10.0 #max trajectory step size in days
inspiral_kwargs = {
    "err":1e-11, #default = 1e-11
    "max_step_size":max_step_days*24*60*60, #in seconds
}
sum_kwargs = {
    "pad_output": True, # True if expecting waveforms smaller than LISA observation window.
}

waveform_class_kwargs = dict(inspiral_kwargs=inspiral_kwargs,
                              mode_selector_kwargs=dict(mode_selection_threshold=1e-5),
                              sum_kwargs=sum_kwargs,
                              use_gpu=use_gpu)

waveform_generator = GenerateEMRIWaveform
waveform_generator_kwargs = dict(return_list=False)



In [3]:
if(use_gpu):
    xp=cp
else:
    xp=np

best_fit_0pa_vs_1pa=  [ 1.38248765e+01,9.98089433e+03,-8.93155110e-01,4.22413106e+01,
                       5.00088445e-01,1.61598675e+00,2.00931361e+00,3.98289416e-01,4.14110626e-01]

best_fit_with_deviation = [ 1.38200811e+01,6.33238062e+03,-8.96151413e-01,4.21591531e+01,4.99958263e-01,
                           1.76591062e+00,1.08140575e+00,1.55045250e-01,3.95573277e-01,-2.28156450e-01,-1.62510561e-03]

#logm1_, m2_, a_, p0_, e0_,qS_,phiS_,Phi_phi0_,Phi_r0_,dev0p_,dev0e_ = params[i]

m1 = 1e6
m2 = 1e4
a = -0.9 # 0.95
p0 = 42.0
e0 = 5.00000000e-01
xI0 = 1.0
dist = 5.0
qS = 1.04719755e+00
phiS = 7.85398163e-01
qK = 6.28318531e-01

phiK = 5.23598776e-01
Phi_phi0 = 0.1
Phi_theta0 =0.2
Phi_r0 = 0.3

dev_0_p = 0
dev_0_e = 0

dt = 10.0
T = 1.0

chi2 = 0.95


dev_1_p=0.0
dev_1_e=0.0
dev_2_p=0.0
dev_2_e=0.0
evolve_1PA = False
evolve_primary = False
evolve_2PA = False
deviation_included=True

emri_kwargs = {"T":T, "dt":dt}


In [4]:
superkludge_wave = GenerateEMRIWaveform(SuperKludgeWaveform,\
                                    sum_kwargs=sum_kwargs,\
                                    return_list=True,
                                    mode_selector_kwargs=dict(mode_selection_threshold=1e-5),
                                    inspiral_kwargs=inspiral_kwargs,
                                    use_gpu=use_gpu)

In [5]:
add_args = [chi2, True, evolve_primary, evolve_2PA,deviation_included,0,0,0,0,0,0]
waveform_true = superkludge_wave(m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0, *add_args, dt=dt, T=T)
waveform_true= xp.asarray(waveform_true)

In [6]:
logm1_bf = best_fit_with_deviation[0]
m1_bf = np.exp(logm1_bf)
m2_bf = best_fit_with_deviation[1]
a_bf = best_fit_with_deviation[2]
p0_bf = best_fit_with_deviation[3]
e0_bf  = best_fit_with_deviation[4]
qS_bf = best_fit_with_deviation[5]
phiS_bf = best_fit_with_deviation[6]
Phi_phi0_bf = best_fit_with_deviation[7]
Phi_r0_bf = best_fit_with_deviation[8]
dev_0_p_bf =best_fit_with_deviation[9]
dev_0_e_bf =best_fit_with_deviation[10]

evolve_1PA_bf = False
deviation_included_bf = True

add_args_bf = [chi2, evolve_1PA_bf, evolve_primary, evolve_2PA,deviation_included_bf,dev_0_p_bf,dev_0_e_bf,dev_1_p,dev_1_e,dev_2_p,dev_2_e]
waveform_bf = superkludge_wave(m1_bf, m2_bf, a_bf, p0_bf, e0_bf, xI0, dist, qS_bf, phiS_bf, qK, phiK, Phi_phi0_bf, Phi_theta0, Phi_r0_bf, *add_args_bf, dt=dt, T=T)
waveform_bf_1= xp.asarray(waveform_bf)

In [7]:
PSD=generate_PSD(waveform_true,dt,use_gpu=use_gpu,
                noise_PSD=get_sensitivity,
                noise_kwargs={'sens_fn':CornishLISASens,'return_type':'PSD'},
                channels=["A","E"])
diff_inner_bf=inner_product(waveform_bf,waveform_true,PSD,dt,use_gpu=use_gpu)
norm_true=np.sqrt(inner_product(waveform_true,waveform_true,PSD,dt,use_gpu=use_gpu))
norm_bf=np.sqrt(inner_product(waveform_bf,waveform_bf,PSD,dt,use_gpu=use_gpu))
overlap_bf=diff_inner_bf/((norm_true)*norm_bf)
print("overlap between best fit point and true waveform:", overlap_bf)
print("mismatch between best fit point and true waveform:", 1-overlap_bf)


overlap between best fit point and true waveform: 0.005658946534629341
mismatch between best fit point and true waveform: 0.9943410534653706


In [8]:
best_fit_with_deviation = [ 1.38160766e+01,7.73822653e+03,-9.00403425e-01,4.20592997e+01,5.00023850e-01,
                           1.60761888e+00,1.82033956e+00,2.62943353e-01,2.14856425e-01,-1.53535661e-01,2.34228475e-02]
logm1_bf = best_fit_with_deviation[0]
m1_bf = np.exp(logm1_bf)
m2_bf = best_fit_with_deviation[1]
a_bf = best_fit_with_deviation[2]
p0_bf = best_fit_with_deviation[3]
e0_bf  = best_fit_with_deviation[4]
qS_bf = best_fit_with_deviation[5]
phiS_bf = best_fit_with_deviation[6]
Phi_phi0_bf = best_fit_with_deviation[7]
Phi_r0_bf = best_fit_with_deviation[8]
dev_0_p_bf =best_fit_with_deviation[9]
dev_0_e_bf =best_fit_with_deviation[10]

evolve_1PA_bf = False
deviation_included_bf = True

add_args_bf = [chi2, evolve_1PA_bf, evolve_primary, evolve_2PA,deviation_included_bf,dev_0_p_bf,dev_0_e_bf,dev_1_p,dev_1_e,dev_2_p,dev_2_e]
waveform_bf = superkludge_wave(m1_bf, m2_bf, a_bf, p0_bf, e0_bf, xI0, dist, qS_bf, phiS_bf, qK, phiK, Phi_phi0_bf, Phi_theta0, Phi_r0_bf, *add_args_bf, dt=dt, T=T)
waveform_bf_2= xp.asarray(waveform_bf)

In [9]:
diff_inner_bf=inner_product(waveform_bf,waveform_true,PSD,dt,use_gpu=use_gpu)
norm_true=np.sqrt(inner_product(waveform_true,waveform_true,PSD,dt,use_gpu=use_gpu))
norm_bf=np.sqrt(inner_product(waveform_bf,waveform_bf,PSD,dt,use_gpu=use_gpu))
overlap_bf=diff_inner_bf/((norm_true)*norm_bf)
print("overlap between best fit point and true waveform:", overlap_bf)
print("mismatch between best fit point and true waveform:", 1-overlap_bf)

overlap between best fit point and true waveform: 0.009894015461101779
mismatch between best fit point and true waveform: 0.9901059845388982


### Overview
Same mismatch we found,
for other points while sampling
## array([-1143.85089403])
### [ 1.38160766e+01,7.73822653e+03,-9.00403425e-01,4.20592997e+01,5.00023850e-01,1.60761888e+00,1.82033956e+00,2.62943353e-01,2.14856425e-01,-1.53535661e-01,2.34228475e-02]

## overlap -> 0.009894015461101779

## array([-1142.58611995]) (*best point*)

### [ 1.38200811e+01,6.33238062e+03,-8.96151413e-01,4.21591531e+01,4.99958263e-01,1.76591062e+00,1.08140575e+00,1.55045250e-01,3.95573277e-01,-2.28156450e-01,-1.62510561e-03]

## overlap -> 0.005658946534629341

In [10]:
logm1_0v1_bf = best_fit_0pa_vs_1pa[0]
m1_0v1_bf = np.exp(logm1_0v1_bf)
m2_0v1_bf = best_fit_0pa_vs_1pa[1]
a_0v1_bf = best_fit_0pa_vs_1pa[2]
p0_0v1_bf = best_fit_0pa_vs_1pa[3]
e0_0v1_bf  = best_fit_0pa_vs_1pa[4]
qS_0v1_bf = best_fit_0pa_vs_1pa[5]
phiS_0v1_bf = best_fit_0pa_vs_1pa[6]
Phi_phi0_0v1_bf = best_fit_0pa_vs_1pa[7]
Phi_r0_0v1_bf = best_fit_0pa_vs_1pa[8]
dev_0_p_bf = 0 
dev_0_e_bf = 0

evolve_1PA_0v1_bf = False
deviation_included_0v1_bf = False

add_args_0v1_bf = [chi2, evolve_1PA_0v1_bf, evolve_primary, evolve_2PA,deviation_included_0v1_bf,
                   dev_0_p_bf,dev_0_e_bf,dev_1_p,dev_1_e,dev_2_p,dev_2_e]
waveform_0v1_bf = superkludge_wave(m1_0v1_bf, m2_0v1_bf, a_0v1_bf, p0_0v1_bf, e0_0v1_bf, xI0, dist, qS_0v1_bf, phiS_0v1_bf, qK, phiK, Phi_phi0_0v1_bf, Phi_theta0, Phi_r0_0v1_bf, *add_args_0v1_bf, dt=dt, T=T)
waveform_0v1_bf = xp.asarray(waveform_0v1_bf)

In [11]:
PSD=generate_PSD(waveform_true,dt,use_gpu=use_gpu,
                noise_PSD=get_sensitivity,
                noise_kwargs={'sens_fn':CornishLISASens,'return_type':'PSD'},
                channels=["A","E"])
diff_inner_bf=inner_product(waveform_0v1_bf,waveform_true,PSD,dt,use_gpu=use_gpu)
norm_true=np.sqrt(inner_product(waveform_true,waveform_true,PSD,dt,use_gpu=use_gpu))
norm_bf=np.sqrt(inner_product(waveform_0v1_bf,waveform_0v1_bf,PSD,dt,use_gpu=use_gpu))
overlap_bf=diff_inner_bf/((norm_true)*norm_bf)
print("overlap between best fit point and true waveform:", overlap_bf)
print("mismatch between best fit point and true waveform:", 1-overlap_bf)


overlap between best fit point and true waveform: 0.006937265780691638
mismatch between best fit point and true waveform: 0.9930627342193084


In [12]:
best_fit_0pa_vs_1pa= [ 1.38209472e+01,9.99442774e+03,-8.98428095e-01,4.21490003e+01,5.00018262e-01,
                      1.64633433e+00,1.91272385e+00,-6.77697076e-02,3.40396819e-01]
logm1_0v1_bf = best_fit_0pa_vs_1pa[0]
m1_0v1_bf = np.exp(logm1_0v1_bf)
m2_0v1_bf = best_fit_0pa_vs_1pa[1]
a_0v1_bf = best_fit_0pa_vs_1pa[2]
p0_0v1_bf = best_fit_0pa_vs_1pa[3]
e0_0v1_bf  = best_fit_0pa_vs_1pa[4]
qS_0v1_bf = best_fit_0pa_vs_1pa[5]
phiS_0v1_bf = best_fit_0pa_vs_1pa[6]
Phi_phi0_0v1_bf = best_fit_0pa_vs_1pa[7]
Phi_r0_0v1_bf = best_fit_0pa_vs_1pa[8]
dev_0_p_bf = 0 
dev_0_e_bf = 0

evolve_1PA_0v1_bf = False
deviation_included_0v1_bf = False

add_args_0v1_bf = [chi2, evolve_1PA_0v1_bf, evolve_primary, evolve_2PA,deviation_included_0v1_bf,
                   dev_0_p_bf,dev_0_e_bf,dev_1_p,dev_1_e,dev_2_p,dev_2_e]
waveform_0v1_bf = superkludge_wave(m1_0v1_bf, m2_0v1_bf, a_0v1_bf, p0_0v1_bf, e0_0v1_bf, xI0, dist, qS_0v1_bf, phiS_0v1_bf, qK, phiK, Phi_phi0_0v1_bf, Phi_theta0, Phi_r0_0v1_bf, *add_args_0v1_bf, dt=dt, T=T)
waveform_0v1_bf = xp.asarray(waveform_0v1_bf)

In [13]:
diff_inner_bf=inner_product(waveform_0v1_bf,waveform_true,PSD,dt,use_gpu=use_gpu)
norm_true=np.sqrt(inner_product(waveform_true,waveform_true,PSD,dt,use_gpu=use_gpu))
norm_bf=np.sqrt(inner_product(waveform_0v1_bf,waveform_0v1_bf,PSD,dt,use_gpu=use_gpu))
overlap_bf=diff_inner_bf/((norm_true)*norm_bf)
print("overlap between best fit point and true waveform:", overlap_bf)
print("mismatch between best fit point and true waveform:", 1-overlap_bf)

overlap between best fit point and true waveform: 0.010783775199385132
mismatch between best fit point and true waveform: 0.9892162248006149


## array([-1183.16159547]) 
### [ 1.38209472e+01,9.99442774e+03,-8.98428095e-01,4.21490003e+01,5.00018262e-01,1.64633433e+00,1.91272385e+00,-6.77697076e-02,3.40396819e-01]
## overlap -> 0.010783775199385132
## array([-1173.08449781]) (*best-fit point*)
### [ 1.38248765e+01,9.98089433e+03,-8.93155110e-01,4.22413106e+01,5.00088445e-01,1.61598675e+00,2.00931361e+00,3.98289416e-01,4.14110626e-01]
## overlap -> 0.006937265780691638